[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/2_nlu.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

# Minimal pipeline - Connecting the components

This notebook runs a minimal end-to-end pipeline that stitches together the components from the notebooks:

1. NLU (intent + slots extraction)
2. DM (select next-best-action)
3. NLG (lexicalize the action into natural language answers)

Each step uses the same helper utilities shown in the other notebooks and is intentionally small so you can run and inspect each stage.

For the final project, you are expected to 
- manage this across multiple turns
- include the [Dialogue State Tracker](./2_nlu.ipynb)


In [2]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.93it/s]


In [3]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

nlu = """Identify the user intent from this list: 
[pizza_ordering, drink_ordering, out_of_domain].

If the intent is pizza_ordering, extract the following slot values from the user input
- pizza_size, the size of the pizza
- pizza_type, the type of pizza
- pizza_count, the number of pizzas as a numeric value.
If no values are present in the user input you have to put 'null' as value.

Only output the json object without any additional text.
The json format is: 
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}"""

dm = """You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the provided dialogue state, select the next best action from the list below:
- request_info(slot), if a slot value is missing (i.e., null)
- provide_info(intent, slot), provide the list of possible values for the requested slot for the given intent
- confirmation(intent), if all slots have been filled

Substitute 'slot' and 'intent' with the actual slot name and intent name from the dialogue state.
Only respond with the action in the exact format specified above. Do not include any additional text or explanation.
"""

nlg = """You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
"""

In [4]:
import torch
import json

user_utterance = "I would like two margherita pizzas for delivery, please."

messages = [{"role": "system", "content": nlu}]
user_msg = user_utterance
text = prepare_text(user_msg, tokenizer, messages)

inputs = tokenizer([text], return_tensors="pt").to(model.device)
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256).cpu()

# decode and try to parse JSON
output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()
nlu_out = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

nlu_data = json.loads(nlu_out)
print(json.dumps(nlu_data, indent=2))

{
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "null",
    "pizza_type": "margherita",
    "pizza_count": 2
  }
}


In [5]:
messages = [{"role": "system", "content": dm}]
text = prepare_text(nlu_out, tokenizer, messages)

inputs = tokenizer([text], return_tensors="pt").to(model.device)
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256).cpu()

# decode and try to parse JSON
output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()
dm_out = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
dm_out

'request_info(pizza_size)'

In [6]:
messages = [{"role": "system", "content": nlg}]
text = prepare_text(f"NBA: {dm_out}\nDS: {nlu_out}", tokenizer, messages)
inputs = tokenizer([text], return_tensors="pt").to(model.device)
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256).cpu()

output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()
nlg_out = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
nlg_out

'Could you please let me know your preferred pizza size? We have options like small, medium, large, or extra large. Your current choice is margherita with 2 pizzas.'